### DATA INGESTION

In [1]:
import os
%pwd

'c:\\Users\\Korisnik\\Desktop\\ml_projects\\summerizer\\TextSummarizer\\research'

In [2]:
os.chdir("../")
%pwd

'c:\\Users\\Korisnik\\Desktop\\ml_projects\\summerizer\\TextSummarizer'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig: 
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [4]:
from src.textsummarizer.constants import *
from src.textsummarizer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(self, config_path=CONFIG_FILE_PATH, params_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_file_path)

        create_directories([self.config.artifact_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )

        return data_ingestion_config

In [6]:
import os
import urllib.request as request
import zipfile

from src.textsummarizer.logging import logger

In [7]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):

        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )

            logger.info(f"File is downloaded")
        else:
            logger.info(f'File already exists.')

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok = True)

        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [8]:
config = ConfigurationManager()
data_ingestion_config = config.get_data_ingestion_config()
data_ingestion = DataIngestion(config = data_ingestion_config)

data_ingestion.download_file()
data_ingestion.extract_zip_file()

[2026-05-18 22:31:16,159: INFO: common]: YAML file config\config.yaml loaded successfully.
[2026-05-18 22:31:16,162: INFO: common]: YAML file params.yaml loaded successfully.
[2026-05-18 22:31:16,163: INFO: common]: Directory created at: artifacts
[2026-05-18 22:31:16,164: INFO: common]: Directory created at: artifacts/data_ingestion
[2026-05-18 22:31:22,709: INFO: 2283541848]: File is downloaded
